# MicroGrad Demo

Scalar autograd engine (`src/engine.py`) and neural network API (`src/nn.py`) built from scratch, then used to train a small MLP on the two moons dataset.

## 1. Setup

Run from the repository root. On Colab, set `repoUrl` and `branchName` to the submission fork and the cell clones it.

In [ ]:
import os
import sys
import subprocess

repoUrl = "https://github.com/YOUR-USER/assignment-1-micrograd-template.git"
branchName = "solution"

if not os.path.isdir("src"):
    subprocess.run(["git", "clone", "-q", "-b", branchName, repoUrl, "micrograd"], check=True)
    os.chdir("micrograd")

sys.path.insert(0, os.getcwd())
os.makedirs("figures", exist_ok=True)
print(os.getcwd())

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

from src.engine import Value
from src.nn import Neuron, Layer, MLP

## 2. Autograder test suite

In [ ]:
testRun = subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider"], capture_output=True, text=True, env={**os.environ, "PYTHONPATH": os.getcwd()})
print(testRun.stdout[-3000:])

## 3. Gradient checks

First the hand-derived example $f = ab(a+b)$ at $a=-2, b=3$, where $\partial f/\partial a = 2ab + b^2 = -3$ and $\partial f/\partial b = a^2 + 2ab = -8$.

In [ ]:
a = Value(-2.0, label="a")
b = Value(3.0, label="b")
f = (a * b) * (a + b)
f.backward()
print(f"f = {f.data}")
print(f"df/da = {a.grad}  (expected -3.0)")
print(f"df/db = {b.grad}  (expected -8.0)")

x = Value(3.0)
diamond = (x + x) * x
diamond.backward()
print(f"diamond d = 2x^2, dd/dx = {x.grad}  (expected 12.0)")

Then every operator together, compared with central finite differences $\frac{g(x+h) - g(x-h)}{2h}$.

In [ ]:
def mixedExpression(xData, yData):
    x = Value(xData)
    y = Value(yData)
    out = (x * y + x ** 2).tanh() - (y / x).exp() + 3 - 2 * y ** 3 / (1 + x ** 2)
    return out, x, y

stepSize = 1e-6
for xData, yData in [(0.5, -1.2), (1.7, 0.3), (-0.8, 2.1)]:
    out, x, y = mixedExpression(xData, yData)
    out.backward()
    numericX = (mixedExpression(xData + stepSize, yData)[0].data - mixedExpression(xData - stepSize, yData)[0].data) / (2 * stepSize)
    numericY = (mixedExpression(xData, yData + stepSize)[0].data - mixedExpression(xData, yData - stepSize)[0].data) / (2 * stepSize)
    print(f"x={xData:5.2f} y={yData:5.2f} | dx autograd {x.grad: .6f} numeric {numericX: .6f} | dy autograd {y.grad: .6f} numeric {numericY: .6f}")

## 4. Data

100 points from two interleaving half circles, labels mapped to $\{-1, +1\}$ to match the tanh output.

In [ ]:
random.seed(1337)
np.random.seed(1337)

points, labels = make_moons(n_samples=100, noise=0.1, random_state=1)
inputs = [[float(px), float(py)] for px, py in points]
targets = [float(label * 2 - 1) for label in labels]

print("inputs:", len(inputs), "first:", inputs[0], "target:", targets[0])

plt.figure(figsize=(5, 5))
plt.scatter(points[:, 0], points[:, 1], c=labels, s=20, cmap="coolwarm")
plt.title("Two moons dataset")
plt.savefig("figures/dataset.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Model

MLP with two hidden layers of 16 tanh neurons and one tanh output.

In [ ]:
model = MLP(2, [16, 16, 1])
print("layers:", len(model.layers), "parameters:", len(model.parameters()))

## 6. Training

Full-batch gradient descent on mean squared error plus a small L2 penalty, learning rate decayed linearly from 1.0 to 0.1.

In [ ]:
def computeLoss(model, inputs, targets, regStrength=1e-4):
    predictions = [model(point) for point in inputs]
    dataLoss = sum((pred - target) ** 2 for pred, target in zip(predictions, targets)) * (1.0 / len(targets))
    regLoss = regStrength * sum(param * param for param in model.parameters())
    accuracy = sum((pred.data > 0) == (target > 0) for pred, target in zip(predictions, targets)) / len(targets)
    return dataLoss + regLoss, accuracy

stepCount = 100
lossHistory = []
accuracyHistory = []

for step in range(stepCount):
    totalLoss, accuracy = computeLoss(model, inputs, targets)
    model.zero_grad()
    totalLoss.backward()
    learningRate = 1.0 - 0.9 * step / stepCount
    for param in model.parameters():
        param.data -= learningRate * param.grad
    lossHistory.append(totalLoss.data)
    accuracyHistory.append(accuracy)
    if step % 10 == 0 or step == stepCount - 1:
        print(f"step {step:3d}  loss {totalLoss.data:.4f}  accuracy {accuracy * 100:.1f}%")

In [ ]:
fig, (lossAxis, accuracyAxis) = plt.subplots(1, 2, figsize=(10, 4))
lossAxis.plot(lossHistory)
lossAxis.set_xlabel("step")
lossAxis.set_ylabel("loss")
lossAxis.set_title("Training loss")
accuracyAxis.plot([value * 100 for value in accuracyHistory])
accuracyAxis.set_xlabel("step")
accuracyAxis.set_ylabel("accuracy (%)")
accuracyAxis.set_title("Training accuracy")
plt.tight_layout()
plt.savefig("figures/training.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Decision boundary

In [ ]:
gridStep = 0.1
xMin, xMax = points[:, 0].min() - 0.5, points[:, 0].max() + 0.5
yMin, yMax = points[:, 1].min() - 0.5, points[:, 1].max() + 0.5
gridX, gridY = np.meshgrid(np.arange(xMin, xMax, gridStep), np.arange(yMin, yMax, gridStep))
gridScores = np.array([model([float(px), float(py)]).data for px, py in zip(gridX.ravel(), gridY.ravel())])

plt.figure(figsize=(6, 5))
plt.contourf(gridX, gridY, (gridScores > 0).reshape(gridX.shape), cmap="coolwarm", alpha=0.6)
plt.scatter(points[:, 0], points[:, 1], c=labels, s=20, cmap="coolwarm", edgecolors="k")
plt.title(f"Decision boundary (accuracy {accuracyHistory[-1] * 100:.1f}%)")
plt.savefig("figures/decisionBoundary.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Results

In [ ]:
summaryLines = [
    f"parameters: {len(model.parameters())}",
    f"initial loss: {lossHistory[0]:.4f}",
    f"final loss: {lossHistory[-1]:.4f}",
    f"initial accuracy: {accuracyHistory[0] * 100:.1f}%",
    f"final accuracy: {accuracyHistory[-1] * 100:.1f}%",
]
with open("figures/results.txt", "w") as resultsFile:
    resultsFile.write("\n".join(summaryLines) + "\n")
print("\n".join(summaryLines))